# Feature Distillation on top of DKD

**Setup:** frozen pretrained **ResNet-56** CIFAR-10 teacher (`torch.hub`, `chenyaofo/pytorch-cifar-models`), ViT student, metric = **wall-clock to 80%** test accuracy (same protocol as the DKD notebook).

**What's new.** DKD uses **logits only**. Here we additionally match the teacher's **penultimate features** (output of ResNet's global average pool, before its `fc`) to the student's **`[CLS]` token** (output of the final `LayerNorm`, before the classification `head`) through a **small learnable linear projection**.

The total loss is

$$
\mathcal{L} = w_{\text{CE}} \cdot \mathrm{CE} + \mathrm{ramp}(t) \cdot \bigl(\alpha \cdot \mathrm{TCKD} + \beta \cdot \mathrm{NCKD}\bigr) + w_{\text{feat}} \cdot \mathrm{Feat}
$$

where $\mathrm{Feat} = \mathrm{MSE}\bigl(\mathrm{proj}(\mathrm{cls}_{\text{student}}),\; f_{\text{teacher}}^{\text{penult}}\bigr)$ (or normalized cosine) and the feature target is `.detach()`'d.

**Hypothesis.** Feature matching gives a denser, per-sample regression signal than a 10-way softmax, so the student's representation aligns with the CNN teacher's earlier in training — reducing **epochs-to-target** and therefore **wall-clock to 80%**.

## Imports

In [ ]:
import sys
import math
import time
import importlib.util
from pathlib import Path
from typing import Dict, List, Optional

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data"

import torch
import torch.nn as nn
import torch.nn.functional as F

from src.model import count_parameters
from src.utils import (
    get_device,
    get_cifar10_loaders,
    get_model,
    set_seed,
    validate,
    get_peak_gpu_memory_mb,
    reset_peak_gpu_memory,
)

torch.set_float32_matmul_precision("high")
print(torch.__version__, get_device())

## Config

DKD hyperparameters match the DKD notebook. New knobs:

- `FEAT_WEIGHT` — scale on the feature-matching loss.
- `FEAT_LOSS` — `"mse"` or `"cos"` (cosine similarity, normalized features).
- `FEAT_WARMUP_EPOCHS` — ramp for the feature term, same shape as DKD ramp.

In [ ]:
SEED = 42
TARGET_ACC = 80.0
MAX_EPOCHS = 100

BATCH_SIZE = 512
LR_REF_BATCH = 512
LR_REF = 1e-3
LR = LR_REF * (BATCH_SIZE / LR_REF_BATCH)
NUM_WORKERS = 4
WARMUP_EPOCHS = 5

# DKD (same defaults as DKD notebook)
TEMP_DKD = 4.0
ALPHA_DKD = 1.0
BETA_DKD = 8.0
CE_WEIGHT = 1.0
DKD_WARMUP_EPOCHS = 20

# Feature distillation
FEAT_WEIGHT = 4.0            # tune 1-10; start at 4
FEAT_LOSS = "mse"            # "mse" or "cos"
FEAT_WARMUP_EPOCHS = 20

VALIDATE_EVERY_COARSE = 2
VALIDATE_DENSE_AFTER = 74.0

set_seed(SEED, deterministic=False)
print(
    f"batch={BATCH_SIZE} LR={LR:.2e} DKD T={TEMP_DKD} a={ALPHA_DKD} b={BETA_DKD} "
    f"CE_w={CE_WEIGHT} FEAT_w={FEAT_WEIGHT} ({FEAT_LOSS})"
)

## Data

In [ ]:
train_loader, test_loader = get_cifar10_loaders(
    data_dir=str(DATA_DIR),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    augment_train=True,
    use_randaugment=True,
    randaugment_num_ops=2,
    randaugment_magnitude=9,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2,
)
xb, _ = next(iter(train_loader))
device = get_device()
print(len(train_loader), "batches", xb.shape)

## DKD loss (Megvii reference)

In [ ]:
def _get_gt_mask(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    target = target.reshape(-1)
    return torch.zeros_like(logits).scatter_(1, target.unsqueeze(1), 1).bool()


def _get_other_mask(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    target = target.reshape(-1)
    return torch.ones_like(logits).scatter_(1, target.unsqueeze(1), 0).bool()


def cat_mask(t: torch.Tensor, mask1: torch.Tensor, mask2: torch.Tensor) -> torch.Tensor:
    t1 = (t * mask1).sum(dim=1, keepdim=True)
    t2 = (t * mask2).sum(dim=1, keepdim=True)
    return torch.cat([t1, t2], dim=1)


def dkd_loss(
    logits_student: torch.Tensor,
    logits_teacher: torch.Tensor,
    target: torch.Tensor,
    alpha: float,
    beta: float,
    temperature: float,
) -> torch.Tensor:
    gt_mask = _get_gt_mask(logits_student, target)
    other_mask = _get_other_mask(logits_student, target)
    pred_student = F.softmax(logits_student / temperature, dim=1)
    pred_teacher = F.softmax(logits_teacher / temperature, dim=1)
    pred_student = cat_mask(pred_student, gt_mask, other_mask)
    pred_teacher = cat_mask(pred_teacher, gt_mask, other_mask)
    log_pred_student = torch.log(pred_student.clamp_min(1e-12))
    tckd = F.kl_div(log_pred_student, pred_teacher, reduction="sum") * (temperature ** 2) / target.shape[0]

    pred_teacher_nt = F.softmax(logits_teacher / temperature - 1000.0 * gt_mask.float(), dim=1)
    log_pred_student_nt = F.log_softmax(logits_student / temperature - 1000.0 * gt_mask.float(), dim=1)
    nckd = F.kl_div(log_pred_student_nt, pred_teacher_nt, reduction="sum") * (temperature ** 2) / target.shape[0]

    return alpha * tckd + beta * nckd


print("DKD helpers ready.")

## Feature hooks and projection

**Teacher feature** = output of ResNet-56's global average pool (the input to its final `fc`). For `chenyaofo/pytorch-cifar-models` this is the `avgpool` module.

**Student feature** = `[CLS]` token after the final `LayerNorm`, i.e. the **input** to `student.head`. We capture it with a `register_forward_pre_hook` on `student.head`.

A small learnable linear `proj: R^{d_s} -> R^{d_t}` aligns the student's 192-d feature with the teacher's 64-d feature before the loss.

In [ ]:
class FeatureGrabber:
    """Latch for the most recent tensor produced by a module (via forward hook)."""

    def __init__(self) -> None:
        self.feat: Optional[torch.Tensor] = None
        self._handle = None

    def attach_post(self, module: nn.Module) -> "FeatureGrabber":
        def hook(_m, _inp, out):
            self.feat = out
        self._handle = module.register_forward_hook(hook)
        return self

    def attach_pre(self, module: nn.Module) -> "FeatureGrabber":
        def hook(_m, inp):
            self.feat = inp[0]
        self._handle = module.register_forward_pre_hook(hook)
        return self

    def detach(self) -> None:
        if self._handle is not None:
            self._handle.remove()
            self._handle = None


def feat_loss_fn(student_feat: torch.Tensor, teacher_feat: torch.Tensor, kind: str) -> torch.Tensor:
    # teacher_feat may be (B, C, 1, 1) from avgpool -> flatten
    if teacher_feat.dim() == 4:
        teacher_feat = teacher_feat.flatten(1)
    teacher_feat = teacher_feat.detach()
    if kind == "mse":
        return F.mse_loss(student_feat, teacher_feat)
    if kind == "cos":
        s = F.normalize(student_feat, dim=-1)
        t = F.normalize(teacher_feat, dim=-1)
        return (1.0 - (s * t).sum(dim=-1)).mean()
    raise ValueError(f"Unknown FEAT_LOSS={kind}")


print("Feature helpers ready.")

## Teacher + student + hooks

In [ ]:
device = get_device()
cnn_teacher = torch.hub.load(
    "chenyaofo/pytorch-cifar-models",
    "cifar10_resnet56",
    pretrained=True,
)
cnn_teacher.eval()
for p in cnn_teacher.parameters():
    p.requires_grad_(False)
cnn_teacher = cnn_teacher.to(device)
_, tacc = validate(cnn_teacher, test_loader, nn.CrossEntropyLoss(), device)
print(f"Teacher test {tacc:.2f}%")

# Discover teacher's penultimate layer: prefer `avgpool`, else register on input to `fc`.
teacher_feat_module = None
for name in ("avgpool", "global_pool", "gap"):
    if hasattr(cnn_teacher, name):
        teacher_feat_module = getattr(cnn_teacher, name)
        print(f"Teacher feature hook: post-hook on .{name}")
        break

teacher_grab = FeatureGrabber()
if teacher_feat_module is not None:
    teacher_grab.attach_post(teacher_feat_module)
else:
    assert hasattr(cnn_teacher, "fc"), "Cannot locate teacher penultimate layer"
    teacher_grab.attach_pre(cnn_teacher.fc)
    print("Teacher feature hook: pre-hook on .fc")

# Probe dimensions with a dummy forward.
with torch.no_grad():
    _ = cnn_teacher(xb.to(device)[:2])
tf = teacher_grab.feat
TEACHER_FEAT_DIM = tf.flatten(1).shape[1]
print("teacher feat shape:", tuple(tf.shape), "-> dim", TEACHER_FEAT_DIM)


def make_student():
    return get_model(patch_size=4, embed_dim=192, depth=6, num_heads=6, dropout=0.0)

print("student params", count_parameters(make_student()))

## Train: `CE + ramp * DKD + ramp * FEAT`

In [ ]:
def train_feat_dkd(student, teacher, train_loader, test_loader):
    device = get_device()
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
    student = student.to(device)
    teacher = teacher.to(device)

    # Student feature = input to head (post-LayerNorm CLS token).
    student_grab = FeatureGrabber().attach_pre(student.head)
    STUDENT_FEAT_DIM = student.head.in_features

    # Linear projection student_feat (d_s) -> teacher_feat (d_t).
    proj = nn.Linear(STUDENT_FEAT_DIM, TEACHER_FEAT_DIM, bias=False).to(device)
    nn.init.trunc_normal_(proj.weight, std=0.02)
    print(f"proj: {STUDENT_FEAT_DIM} -> {TEACHER_FEAT_DIM}")

    # Optional torch.compile (skipped if Triton is missing/incompatible, as on some ARM images).
    _triton_ok = False
    if device.type == "cuda" and importlib.util.find_spec("triton") is not None:
        try:
            from triton.compiler.compiler import triton_key as _tk  # noqa: F401
            _triton_ok = True
        except Exception as e:
            print("torch.compile skipped: incompatible Triton", type(e).__name__, e)
    if _triton_ok:
        try:
            student = torch.compile(student, mode="reduce-overhead")
        except Exception as e:
            print("torch.compile skipped:", type(e).__name__, e)

    opt = torch.optim.AdamW(
        list(student.parameters()) + list(proj.parameters()),
        lr=LR,
        weight_decay=0.01,
    )

    def lr_lambda(ep):
        if ep < WARMUP_EPOCHS:
            return (ep + 1) / WARMUP_EPOCHS
        p = (ep - WARMUP_EPOCHS) / max(1, MAX_EPOCHS - WARMUP_EPOCHS)
        return 0.5 * (1 + math.cos(math.pi * p))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    use_amp = device.type == "cuda"
    dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16

    hist: Dict[str, List] = {
        "test_acc": [],
        "train_loss": [],
        "wall_time": [],
        "feat_loss": [],
    }
    best, ttt = 0.0, None
    last_val: Optional[float] = None
    t0 = time.perf_counter()

    for ep in range(1, MAX_EPOCHS + 1):
        dkd_ramp = min(1.0, ep / max(1, DKD_WARMUP_EPOCHS))
        feat_ramp = min(1.0, ep / max(1, FEAT_WARMUP_EPOCHS))
        student.train()
        tot = 0.0
        feat_tot = 0.0
        t_ep = time.perf_counter()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)

            if use_amp:
                with torch.amp.autocast(device_type="cuda", dtype=dtype):
                    zs = student(x)
                with torch.no_grad():
                    with torch.amp.autocast(device_type="cuda", dtype=dtype):
                        zt = teacher(x)
            else:
                zs = student(x)
                with torch.no_grad():
                    zt = teacher(x)

            s_feat = student_grab.feat
            t_feat = teacher_grab.feat
            assert s_feat is not None and t_feat is not None, "Feature hooks did not fire"

            # Compute losses in fp32 for stability.
            ce = F.cross_entropy(zs.float(), y)
            dkd = dkd_loss(zs.float(), zt.float().detach(), y, ALPHA_DKD, BETA_DKD, TEMP_DKD)
            feat = feat_loss_fn(proj(s_feat.float()), t_feat.float(), FEAT_LOSS)

            loss = CE_WEIGHT * ce + dkd_ramp * dkd + feat_ramp * FEAT_WEIGHT * feat
            loss.backward()
            nn.utils.clip_grad_norm_(
                list(student.parameters()) + list(proj.parameters()), 1.0
            )
            opt.step()
            tot += loss.item()
            feat_tot += feat.item()
        sched.step()
        wt = time.perf_counter() - t_ep

        run_val = (ep == 1) or (ep == MAX_EPOCHS)
        if last_val is not None and last_val >= VALIDATE_DENSE_AFTER:
            run_val = True
        elif ep % VALIDATE_EVERY_COARSE == 0:
            run_val = True
        if run_val:
            _, acc = validate(student, test_loader, nn.CrossEntropyLoss(), device)
            last_val = acc
        else:
            acc = last_val if last_val is not None else 0.0

        best = max(best, acc)
        if ttt is None and acc >= TARGET_ACC:
            ttt = time.perf_counter() - t0
        hist["train_loss"].append(tot / len(train_loader))
        hist["feat_loss"].append(feat_tot / len(train_loader))
        hist["test_acc"].append(acc)
        hist["wall_time"].append(wt)
        tag = "~" if not run_val else ""
        print(
            f"ep{ep:3d} dkd_r={dkd_ramp:.2f} feat_r={feat_ramp:.2f} "
            f"loss {hist['train_loss'][-1]:.4f} feat {hist['feat_loss'][-1]:.4f} "
            f"test{tag} {acc:.2f}% ({wt:.1f}s/ep)"
        )
        if ttt is not None:
            print(f"  >>> {TARGET_ACC}% in {ttt:.2f}s wall-clock")
            break

    student_grab.detach()
    return {
        "history": hist,
        "best_acc": best,
        "time_to_target": ttt,
        "total_time": time.perf_counter() - t0,
    }


set_seed(SEED)
reset_peak_gpu_memory()
results = train_feat_dkd(make_student(), cnn_teacher, train_loader, test_loader)
print(results["time_to_target"], results["best_acc"], "peak", get_peak_gpu_memory_mb())

## Plot

In [ ]:
import matplotlib.pyplot as plt

h = results["history"]
cum = [sum(h["wall_time"][: i + 1]) for i in range(len(h["wall_time"]))]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(h["test_acc"], lw=2)
ax[0].axhline(TARGET_ACC, color="gray", ls="--")
ax[0].set_xlabel("epoch")
ax[0].set_ylabel("test %")
ax[0].set_title("Feature+DKD: test accuracy")
ax[0].grid(alpha=0.3)
ax[1].plot(cum, h["test_acc"], lw=2)
ax[1].axhline(TARGET_ACC, color="gray", ls="--")
if results["time_to_target"]:
    ax[1].axvline(results["time_to_target"], color="r", ls=":", label="hit")
    ax[1].legend()
ax[1].set_xlabel("wall s")
ax[1].set_title("Time-to-target")
ax[1].grid(alpha=0.3)
ax[2].plot(h["feat_loss"], lw=2, color="tab:purple")
ax[2].set_xlabel("epoch")
ax[2].set_ylabel("feature loss")
ax[2].set_title(f"Feature alignment ({FEAT_LOSS})")
ax[2].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "notebooks" / "feature_distillation_results.png", dpi=150, bbox_inches="tight")
plt.show()